In [1]:
# 1-dataset model (HTCas9)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_1_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt: Spearman = 0.3767251588441338
../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt: Spearman = 0.4053121997155018
../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt: Spearman = 0.367213788944335
../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt: Spearman = 0.3845436183573927
../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt: Spearman = 0.370775257283275
../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt: Spearman = 0.37092015562577074
../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt: Spearman = 0.35069255273894445
../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt: Spearman = 0.37816337247534487
../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt: Spearman = 0.3805931070704544
../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt: Spearman = 0.36773713900024635


In [3]:
# 2-dataset model (HTCas9+HT11)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_2_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt: Spearman = 0.4212459608516691
../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt: Spearman = 0.44743607338138286
../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt: Spearman = 0.4478500535759829
../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt: Spearman = 0.4265839142230875
../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt: Spearman = 0.4348658461677295
../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt: Spearman = 0.4372378251092156
../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt: Spearman = 0.4428552075235076
../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt: Spearman = 0.439558305743249
../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt: Spearman = 0.4091889799122594
../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt: Spearman = 0.4398525144391703


In [5]:
# 4-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5))

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_4_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt: Spearman = 0.4268963947153306
../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt: Spearman = 0.4169314824702326
../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt: Spearman = 0.4217558351970149
../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt: Spearman = 0.4253569202194604
../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt: Spearman = 0.4728737473079556
../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt: Spearman = 0.4399328834268326
../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt: Spearman = 0.4300959318017168
../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt: Spearman = 0.41990088841877965
../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt: Spearman = 0.40082825833449204
../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt: Spearman = 0.44159987479252233


In [7]:
# 5-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_5_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt: Spearman = 0.4260608354968791
../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt: Spearman = 0.4264301739947064
../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt: Spearman = 0.4104993517236797
../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt: Spearman = 0.42242028211082194
../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt: Spearman = 0.4184704328277865
../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt: Spearman = 0.41748528803227586
../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt: Spearman = 0.4359610204802313
../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt: Spearman = 0.43362918047207444
../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt: Spearman = 0.3809946410328224
../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt: Spearman = 0.41571622337023656


In [9]:
# 6-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_6_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt: Spearman = 0.4128495320899666
../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt: Spearman = 0.40861354227762825
../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt: Spearman = 0.40559901981144647
../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt: Spearman = 0.4030798168493595
../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt: Spearman = 0.41555547786906705
../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt: Spearman = 0.42175143158457895
../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt: Spearman = 0.42297275273439544
../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt: Spearman = 0.38719366290844365
../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt: Spearman = 0.39295432948629566
../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt: Spearman = 0.41001632038783437


In [11]:
# 7-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER+iMeta)

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_7_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt: Spearman = 0.43002258999892085
../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt: Spearman = 0.42664780373228595
../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt: Spearman = 0.4075944683654201
../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt: Spearman = 0.41614228300776457
../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt: Spearman = 0.3980558474509852
../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt: Spearman = 0.40154936032208044
../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt: Spearman = 0.37609356935407684
../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt: Spearman = 0.41256374415985886
../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt: Spearman = 0.4047262976753134
../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt: Spearman = 0.42423623325221876
